# Exercise on split-apply-combine: compute summary statistics per group

In the previous exercise we combined two tables with a *join*. Now we take the
joined table and compute a summary statistic *per group* using the
split-apply-combine pattern:

1. **Split** the rows into groups (here, by diet group).
2. **Apply** a function to each group (here, sum the cardiovascular events).
3. **Combine** the per-group results into a single table.

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

## Load the patient data

In [2]:
df = pd.read_csv('processed_data_predimed.csv')

In [3]:
df.shape

(6245, 18)

In [4]:
df.head()

,patient-id,location-id,sex,age,smoke,bmi,waist,wth,htn,diab,hyperchol,famhist,hormo,p14,toevent,event,group,City
0,1,1,Female,77,Never,25.92,94,0.657343,Yes,No,Yes,Yes,No,9,5.538672,No,MedDiet + VOO,Madrid
1,2,1,Female,68,Never,34.85,150,0.949367,Yes,No,Yes,Yes,NaN,10,3.063655,No,MedDiet + Nuts,Madrid
2,3,1,Female,66,Never,37.50,120,0.750000,Yes,Yes,No,No,No,6,5.590691,No,MedDiet + Nuts,Madrid
3,4,1,Female,77,Never,29.26,93,0.628378,Yes,Yes,No,No,No,6,5.456537,No,MedDiet + VOO,Madrid
4,5,1,Female,60,Never,30.02,104,0.662420,Yes,No,Yes,No,No,9,2.746064,No,Control,Madrid


## Question: did the mediterranean diet prevent cardiovascular events?

The column `event` contains `Yes` or `No`, indicating whether a patient had a cardiovascular event. 
The column `group` contains which diet the patient followed.

To make the sums easier, we first convert `event` to a binary column (1 for `Yes`, 0 for `No`).

In [5]:
df['event'] = df['event'].map({'Yes': 1, 'No': 0})

Now to answer the question we want the total number of events *per diet group*.

### Naive solution: nested for-loops

A naive way of doing this is as follows. For every group, go through all the rows and add up the events that belong to that group. 

In [6]:
events_nested = {}
for g in df['group'].unique():                                 
    total = 0
    for event, group_label in zip(df['event'], df['group']):   
        if group_label == g:
            total += event                                     
    events_nested[g] = total                                  

events_nested

{'MedDiet + VOO': 83, 'MedDiet + Nuts': 69, 'Control': 96}

- **Q:** Check out the code above. What is this implementation doing?

In [7]:
# answer: 
# for every group, it goes through all the rows and add up the events that belong to that group. 

- **Q:** What is the time complexity of this implementation?

In [8]:
# answer:
# n * g ~~ O(n*g)   with g the number of groups and n the number of rows

Let us measure how long this takes:

In [9]:
%%timeit
# solution
events_nested = {}
for g in df['group'].unique():
    total = 0
    for event, group_label in zip(df['event'], df['group']):
        if group_label == g:
            total += event
    events_nested[g] = total

1.47 ms ± 35 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


### A second solution: iterating through rows

We go through the rows once, and for each row we look up which group it belongs to and add its event to that
group's running total. We keep the running totals in a dictionary.

In [10]:
events_rows = {}
for i, row in df.iterrows():
    g = row['group']
    e = row['event'] # 1 or 0
    if g not in events_rows:
        events_rows[g] = 0
    events_rows[g] += e

events_rows

{'MedDiet + VOO': 83, 'MedDiet + Nuts': 69, 'Control': 96}

Let us measure how long the single pass takes:

In [11]:
%%timeit

events_rows = {}
for i, row in df.iterrows():
    if row['group'] not in events_rows:
        events_rows[row['group']] = 0
    events_rows[row['group']] += row['event']

110 ms ± 3.67 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


Same questions for this implementation.

- What is this implementation doing? and 
- What is the time complexity of this implementation?

In [1]:
# answer:
# it goes through the rows only ONCE. For each row it looks up the group in a
# dictionary and adds the event to that group's running total.
# A dictionary is a hash table, so the look-up and the update cost O(1) on average.
# One pass of n rows, doing O(1) work each --> O(n), independent of g.
#
# So this implementation is asymptotically BETTER than the nested loops, O(n) vs O(n*g).
# And yet it is ~100x SLOWER!  Why? See the discussion at the end of the notebook.

### Exercise: write the fast, one-line version

- Write a single line command that does the same computation with a `split-apply-combine`.
- Check that you get the same numbers as the loops above.
- Time it 

In [13]:
# solution - optimal
events_groupby = df.groupby('group')['event'].sum()
events_groupby

group
Control           96
MedDiet + Nuts    69
MedDiet + VOO     83
Name: event, dtype: int64

In [14]:
%%timeit
# solution
df.groupby('group')['event'].sum()

338 μs ± 9.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


## Answering the question

There were not the same number of patients in each group, so to compare fairly we look at the *percentage* of events per group. 
*Each of these is a one-line split-apply-combine.*

- Print the number of patients per group

In [15]:
# solution - number of patients per group
df.groupby('group')['event'].count()

group
Control           2016
MedDiet + Nuts    2077
MedDiet + VOO     2152
Name: event, dtype: int64

- Calculate the percentage of events per group. 


Hint: the mean of a 0/1 column is the fraction of ones, so multiply by 100 for a percentage

In [16]:
# solution - percentage of events per group
df.groupby('group')['event'].mean() * 100

group
Control           4.761905
MedDiet + Nuts    3.322099
MedDiet + VOO     3.856877
Name: event, dtype: float64

The control group had a higher percentage of events than the two mediterranean
diet groups.

## Further explanation about time complexities

Above we had three implementations:

- Nested for-loops: `O(n * g)`
- Loop over rows with `iterrows`: `O(n)`. We walk through the `n` rows exactly once.
- `groupby`: `O(n)`, uses the same idea as iterrows().

Although `iterrows` and `groupby` both have O(n) complexity, `iterrows` was much slower. Why?


In every iteration, `iterrows` creates a new pandas `Series` object for each row and needs to copy the values into it. This cost time.

`groupby`, on the other hand, never leaves compiled code. It runs everything inside pandas' C implementation, walking contiguous typed arrays rather than executing Python bytecode and allocating objects per row. So although both are O(n), `groupby` is much faster.

The takeaway message is: 
- use built-in pandas pipelines to analyse your data.
- do not iterate through rows. If you have a for loop, you're doing it wrong.
